# DINOv3 Segmentation Frontend
#
**Interactive pipeline for DINO-based unsupervised segmentation.**
#
Stages: Resolution Recovery → [Low-level Fusion] → Clustering →
Assignment → [Post-processing] → Evaluation
#
---
## Cell 0 – Install dependencies



In [ ]:
# Cell 0: Install dependencies
import subprocess, sys

packages = [
    "opencv-python-headless",
    "scikit-learn",
    "scikit-image",
    "scipy",
    "pandas",
    "matplotlib",
    "ipywidgets",
    "pydensecrf @ git+https://github.com/lucasb-eyer/pydensecrf.git",
]

for pkg in packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        check=False,
        capture_output=True,
    )

print("Dependencies installed.")



## Cell 1 – Setup



In [ ]:
# Cell 1: Imports and GPU check
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# GPU check
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# Add dinov3_seg to path
# CONFIG: set this path if the package is in a different location
PACKAGE_PATH = "/kaggle/working/dinov3_seg"
if PACKAGE_PATH not in sys.path:
    sys.path.insert(0, os.path.dirname(PACKAGE_PATH))

from dinov3_seg import Pipeline, get_default_config
from dinov3_seg.config import PipelineConfig
from dinov3_seg.dataloader import get_dataloaders
from dinov3_seg.metrics import compute_all_metrics
from dinov3_seg.utils import denormalize_image, set_seed

set_seed(42)
print("dinov3_seg loaded successfully.")



## Cell 2 – Dataset paths
#
Edit this cell to point to the correct dataset roots on Kaggle.



In [ ]:
# Cell 2: Dataset configuration
# CONFIG: Change dataset paths here
DATASET_PATHS = {
    "Kvasir":  "/kaggle/input/kvasir-seg",
    "BCCD1":   "/kaggle/input/bccd-dataset-1",
    "BCCD2":   "/kaggle/input/bccd-dataset-2",
}

# CONFIG: Change number of semantic classes per dataset
N_CLASSES = {
    "Kvasir": 2,
    "BCCD1":  2,
    "BCCD2":  3,
}



## Cell 3 – Interactive UI



In [ ]:
# Cell 3: Interactive pipeline configuration widgets

# ── Dataset ──────────────────────────────────────────────────────────────────
w_dataset = widgets.Dropdown(
    options=list(DATASET_PATHS.keys()),
    value="Kvasir",
    description="Dataset:",
    style={"description_width": "140px"},
)

w_split = widgets.ToggleButtons(
    options=["train", "test"],
    value="test",
    description="Eval split:",
    style={"description_width": "140px"},
)

# ── Resolution recovery ───────────────────────────────────────────────────────
w_resolution = widgets.Dropdown(
    options=["nearest", "bilinear", "bicubic", "pca", "bilateral"],
    value="bilinear",
    description="Resolution (R):",
    style={"description_width": "140px"},
)

# ── Clustering ────────────────────────────────────────────────────────────────
w_clustering = widgets.Dropdown(
    options=["kmeans", "kmeans_pca", "hdbscan", "spectral",
             "hierarchical", "ncut", "joint_kmeans"],
    value="kmeans",
    description="Clustering (C):",
    style={"description_width": "140px"},
)

# CONFIG: Change K here
w_k = widgets.IntSlider(
    value=8, min=2, max=64, step=1,
    description="K (n_clusters):",
    style={"description_width": "140px"},
)

w_pca_dim = widgets.IntSlider(
    value=32, min=4, max=256, step=4,
    description="PCA dim:",
    style={"description_width": "140px"},
)

w_min_cluster = widgets.IntSlider(
    value=5, min=2, max=50, step=1,
    description="min_cluster_size:",
    style={"description_width": "140px"},
)

# ── Assignment ────────────────────────────────────────────────────────────────
w_assignment = widgets.Dropdown(
    options=["majority_vote", "weighted_majority", "hungarian",
             "label_propagation", "abstention", "cross_image"],
    value="majority_vote",
    description="Assignment (A):",
    style={"description_width": "140px"},
)

# CONFIG: Abstention threshold
w_abstention_thresh = widgets.FloatSlider(
    value=0.6, min=0.0, max=1.0, step=0.05,
    description="Abstention thresh:",
    style={"description_width": "140px"},
)

# ── Post-processing ────────────────────────────────────────────────────────────
w_postprocess = widgets.SelectMultiple(
    options=["none", "morphology", "connected_comp", "dense_crf",
             "superpixel", "bilateral_soft"],
    value=["none"],
    description="Post-process (P):",
    style={"description_width": "140px"},
    layout=widgets.Layout(height="120px"),
)

w_crf_iter = widgets.IntSlider(
    value=10, min=1, max=20, step=1,
    description="CRF iterations:",
    style={"description_width": "140px"},
)

w_min_area = widgets.IntSlider(
    value=200, min=10, max=2000, step=10,
    description="Min CC area:",
    style={"description_width": "140px"},
)

# ── Low-level features ────────────────────────────────────────────────────────
w_lowlevel = widgets.Dropdown(
    options=["none", "color_hist", "hog", "lbp", "edge_avg",
             "slic_pool", "watershed", "late_fusion"],
    value="none",
    description="Low-level (L):",
    style={"description_width": "140px"},
)

# ── Layout ────────────────────────────────────────────────────────────────────
section = lambda title: widgets.HTML(
    f"<h4 style='margin:8px 0 2px 0;border-bottom:1px solid #ccc;'>{title}</h4>"
)

ui = widgets.VBox([
    section("📁 Dataset"),
    w_dataset, w_split,
    section("🔍 Resolution Recovery"),
    w_resolution,
    section("🔵 Clustering"),
    w_clustering, w_k, w_pca_dim, w_min_cluster,
    section("🏷️ Assignment"),
    w_assignment, w_abstention_thresh,
    section("🔧 Post-processing"),
    w_postprocess, w_crf_iter, w_min_area,
    section("🎨 Low-level Features"),
    w_lowlevel,
], layout=widgets.Layout(width="420px"))

display(ui)



## Cell 4 – Build config from widgets



In [ ]:
# Cell 4: Helper to build PipelineConfig from widget values

def build_config() -> PipelineConfig:
    """Construct a PipelineConfig from the current widget values."""
    ds    = w_dataset.value
    cfg   = PipelineConfig(
        dataset=ds,
        dataset_path=DATASET_PATHS[ds],
        n_classes=N_CLASSES[ds],
        device=DEVICE,
        resolution={"method": w_resolution.value},
        clustering=_build_clustering(),
        assignment=_build_assignment(),
        postprocess=_build_postprocess(),
        lowlevel=_build_lowlevel(),
    )
    return cfg


def _build_clustering() -> dict:
    method = w_clustering.value
    cfg = {"method": method, "n_clusters": w_k.value}
    if method == "kmeans_pca":
        cfg["pca_dim"] = w_pca_dim.value
    elif method == "hdbscan":
        cfg["min_cluster_size"] = w_min_cluster.value
    return cfg


def _build_assignment() -> dict:
    method = w_assignment.value
    cfg    = {"method": method}
    if method == "abstention":
        cfg["threshold"] = w_abstention_thresh.value
    return cfg


def _build_postprocess() -> list:
    selected = list(w_postprocess.value)
    if "none" in selected or not selected:
        return []
    pp_list = []
    for name in selected:
        entry = {"method": name}
        if name == "dense_crf":
            entry["n_iter"] = w_crf_iter.value
        elif name == "connected_comp":
            entry["min_area"] = w_min_area.value
        pp_list.append(entry)
    return pp_list


def _build_lowlevel():
    if w_lowlevel.value == "none":
        return None
    return {
        "method":      w_lowlevel.value,
        "fusion_mode": "concat",
        "low_weight":  1.0,
    }


print("Config builder ready.")



## Cell 5 – Run Pipeline button



In [ ]:
# Cell 5: Run Pipeline button

btn_run    = widgets.Button(description="▶ Run Pipeline", button_style="success",
                             layout=widgets.Layout(width="200px"))
btn_ablate = widgets.Button(description="📊 Run Ablation", button_style="info",
                             layout=widgets.Layout(width="200px"))
out        = widgets.Output()

display(widgets.HBox([btn_run, btn_ablate]), out)


def _show_gallery(images, preds, gts, filenames, n=4):
    """Display up to n side-by-side (image | pred | GT) rows."""
    fig, axes = plt.subplots(min(n, len(images)), 3, figsize=(12, 4 * min(n, len(images))))
    if min(n, len(images)) == 1:
        axes = [axes]
    for i, (img_t, pred_t, gt_t, fname) in enumerate(
        zip(images[:n], preds[:n], gts[:n], filenames[:n])
    ):
        img_np = denormalize_image(img_t)
        axes[i][0].imshow(img_np);     axes[i][0].set_title(f"{fname}\nImage")
        axes[i][1].imshow(pred_t.cpu().numpy(), cmap="tab10", vmin=0, vmax=9)
        axes[i][1].set_title("Prediction")
        axes[i][2].imshow(gt_t.cpu().numpy(), cmap="tab10", vmin=0, vmax=9)
        axes[i][2].set_title("Ground Truth")
        for ax in axes[i]: ax.axis("off")
    plt.tight_layout()
    plt.show()


def on_run(_b):
    with out:
        clear_output(wait=True)
        cfg = build_config()
        print(f"Config:\n  Resolution : {cfg.resolution['method']}")
        print(f"  Clustering : {cfg.clustering['method']} K={cfg.clustering.get('n_clusters')}")
        print(f"  Assignment : {cfg.assignment['method']}")
        print(f"  PostProc   : {[p['method'] for p in cfg.postprocess] or 'none'}")
        print(f"  LowLevel   : {cfg.lowlevel['method'] if cfg.lowlevel else 'none'}")
        print(f"\nRunning on {w_split.value} split …")

        try:
            train_loader, test_loader = get_dataloaders(
                cfg.dataset_path, batch_size=1, num_workers=0
            )
            loader = test_loader if w_split.value == "test" else train_loader

            pl = Pipeline(cfg)
            agg_metrics = []
            gallery_imgs, gallery_preds, gallery_gts, gallery_fnames = [], [], [], []

            for batch in loader:
                img_t    = batch["image"][0]
                gt_t     = batch["mask"][0, 0].long()
                tokens_t = batch["patch_tokens"][0]
                fname    = batch["filename"][0] if isinstance(batch["filename"], list) \
                           else batch["filename"][0]

                pred = pl.run(img_t, patch_tokens=tokens_t, gt_mask=gt_t)
                m    = compute_all_metrics(pred, gt_t, cfg.n_classes)
                agg_metrics.append(m)
                gallery_imgs.append(img_t)
                gallery_preds.append(pred)
                gallery_gts.append(gt_t)
                gallery_fnames.append(fname)

            # Aggregate
            keys  = agg_metrics[0].keys()
            final = {}
            for k in keys:
                vals = [m[k] for m in agg_metrics if not np.isnan(m.get(k, float("nan")))]
                final[k] = float(np.mean(vals)) if vals else float("nan")

            # Table
            df = pd.DataFrame([final]).T.rename(columns={0: "Value"})
            df.index.name = "Metric"
            print("\n── Evaluation Results ──")
            display(df.style.format("{:.4f}"))

            # Gallery
            print("\n── Visual Gallery (first 4 samples) ──")
            _show_gallery(gallery_imgs, gallery_preds, gallery_gts, gallery_fnames, n=4)

        except Exception as exc:
            import traceback
            print(f"❌ Error: {exc}")
            traceback.print_exc()


btn_run.on_click(on_run)




## Cell 6 – Ablation sweep



In [ ]:
# Cell 6: Ablation sweep widgets and button

# CONFIG: Change the sweep parameter and range here
w_ablate_param = widgets.Dropdown(
    options=[
        ("K (n_clusters)", "clustering.n_clusters"),
        ("PCA dim",        "clustering.pca_dim"),
        ("Resolution",     "resolution.method"),
    ],
    description="Sweep param:",
    style={"description_width": "140px"},
)

w_ablate_k_min  = widgets.IntText(value=2,  description="K min:", layout=widgets.Layout(width="160px"))
w_ablate_k_max  = widgets.IntText(value=32, description="K max:", layout=widgets.Layout(width="160px"))
w_ablate_k_step = widgets.IntText(value=4,  description="K step:", layout=widgets.Layout(width="160px"))

display(widgets.VBox([
    w_ablate_param,
    widgets.HTML("<i>For K sweep:</i>"),
    widgets.HBox([w_ablate_k_min, w_ablate_k_max, w_ablate_k_step]),
]))


def on_ablate(_b):
    with out:
        clear_output(wait=True)
        cfg    = build_config()
        param  = w_ablate_param.value

        if "n_clusters" in param or "pca_dim" in param:
            values = list(range(w_ablate_k_min.value, w_ablate_k_max.value + 1, w_ablate_k_step.value))
        else:
            values = ["nearest", "bilinear", "bicubic"]

        print(f"Ablation sweep: {param} ∈ {values}")

        try:
            _, test_loader = get_dataloaders(cfg.dataset_path, batch_size=1, num_workers=0)
            pl = Pipeline(cfg)
            df = pl.run_experiment({param: values}, dataloader=test_loader, fixed_config=cfg)

            display(df.style.format("{:.4f}", subset=[c for c in df.columns if c != param]))

            # Plot mIoU vs parameter
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(df[param].astype(str), df["miou"], marker="o", linewidth=2)
            ax.set_xlabel(param)
            ax.set_ylabel("mIoU")
            ax.set_title(f"mIoU vs {param}")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

        except Exception as exc:
            import traceback
            print(f"❌ Error: {exc}")
            traceback.print_exc()


btn_ablate.on_click(on_ablate)



## Cell 7 – Quick-start (no widgets)
#
Run this cell directly for a minimal example without the UI.



In [ ]:
# Cell 7: Quick-start minimal example
# CONFIG: Edit these directly if running without UI

QUICK_DATASET   = "Kvasir"
QUICK_DATASET_PATH = DATASET_PATHS[QUICK_DATASET]
QUICK_N_CLASSES = N_CLASSES[QUICK_DATASET]
QUICK_K         = 8          # CONFIG: Change K here

from dinov3_seg import get_default_config

cfg = get_default_config(
    dataset=QUICK_DATASET,
    dataset_path=QUICK_DATASET_PATH,
    n_classes=QUICK_N_CLASSES,
    device=DEVICE,
)

# Optionally override:
# cfg.clustering["n_clusters"] = QUICK_K
# cfg.resolution["method"] = "bicubic"
# cfg.postprocess = [{"method": "dense_crf", "n_iter": 5}]

pl = Pipeline(cfg)
metrics = pl.evaluate(verbose=True)

print("Done. Metrics:", metrics)

